# IRA PDF Form Build Keys

------------------
## form1065_filler.py
Python class that fills an IRS Form 1065 PDF (U.S. Return of Partnership Income)
from a plain Python dict.


## Workflow: FormKey Mapping

#### A. Gen Key Maps (irs.irsForms)

1. genTestKeyDict
    - irsForm -> fldDict [Field Dict]
    - iraForm -> testDict
    - OUT:
        - testDict
        - FILE: pdfFill(testDict) -> irsForms/<fm>-IRS.pdf.  [visual of forms with F#]
5. genIRSMap
    - fldDict -> keyDict -> irsForms/<fm>-IRS.json [map <fldName> -> F# ]

#### B. Gen IRS Report (irs.pdfFill)

1. gen_glTaxDict
    - glDict -> glKeys. [ Core of IRS Tax mapping]
    - glDict X glKeys -> glTaxDict -> irsForms/<fm>_glKey.json [<fldName -> glDict.alues]
    - glTaxDict X keyDict -> fmDict  [form values: fmKey -> glDict.values]
1. FILE: pdfFill(fmDict) ->  YE_Tax_Report/<fm>-IRS.pdf

------
## Build Key

- input IRS_Forms/formFN.pdf
- get all fields into Dict
- save 

In [51]:
# LLC GL data
import os
from pathlib import Path
import json
from ledger.LLC import LLC
from irs.pdfFill import pdfFill
from IPython.display import display, Markdown

# Link to LLC ledgers
top = Path.cwd().parents[2]
llcName = [f for f in os.listdir(top) if 'llcProfile' in f][0].replace('.json','').replace('llcProfile_','')
llc = LLC(llcName,debug=False, top=top)
# Save entity information
eDict = llc.entity
acctDIR = llc.acctDir() #os.path.join(llc.TOP, llc.dirAccounting, str(llc.yr))
yeDIR = llc.acctDir(dirName='ye')


# ----------   initialize IRS forms
from irs.irsForms import irsF1065, irsSchK1
fmSchK1 = irsSchK1(llc)
fm1065 = irsF1065(llc)

fm = fmSchK1
display(Markdown(f"##### \n## Notebook using:\n### LLC: {llc.entity['entity_name']}\n\n### IRS Form: {fm.oID}"))


##### 
## Notebook using:
### LLC: W&B Group, LLC

### IRS Form: irsSchK1

## Per Tax Year - Do once

In [52]:
# Per Year, Save Form json field names, modify if any changes, json save per year
if False: 
    # Do this once per year to reconcile new forms to IRS definitions.
    from irs.irsFormFieldNames import irsF1065Fields, irsSchKFields
    fmSchK1.fldNmSave(irsSchKFields().fldNmDict)
    fm1065.fldNmSave(irsF1065Fields().fldNmDict)

## Get Dict of all fields in form :: f

In [57]:
fm.genTestKeyDict()

Loading Form Field Name (json): Schedule_K_1-FieldNames.json
DIR: /Users/frankrojas/GDrive/Family/Assets-Hobby/RealEstateInvestments/LLC-WB-Group/pages/AccountingData/2025/YE_Tax_Records/Forms_IRS/Schedule_K_1-FieldNames.json
irsSchK1 SUCC: testDict Generated 111 Fields; 
-- FILE: /Users/frankrojas/GDrive/Family/Assets-Hobby/RealEstateInvestments/LLC-WB-Group/pages/AccountingData/2025/YE_Tax_Records/Forms_IRS/Schedule_K_1-keys.json


In [56]:
irsFormDir = os.path.join(yeDIR, 'Forms_IRS')


pf = pdfFill(fm.inFN(), fm.outFN())
fDict = pf.get()

# Load from json
print("Json FN", fm.FN('FieldNames.json'))
fldNmDict = fm.fldNmLoad()
# Load from py
#fldNmDict = irsF1065Fields().fldNmDict
print(len(fldNmDict), len(fDict))
[k for k in enumerate(fldNmDict)]

def genTestKeyDict(self):

    # 1. create raw dict for every field in form
    #pf = pdfFill(self.inFN(), self.outFN())
    fDict = self.pf.get()

    # 2. Create testDict to map field ID (F#) into every field
    fldNmDict = self.fldNmLoad()
    testDict =  {}
    for (i,k1),(j,k2) in zip(enumerate(fDict), enumerate(fldNmDict)):
        d = fDict[k1]
        d['field_id'] = k2
        testDict[k1] = d
            
    # OLD testDict = {k:self.pf._testKey(i,k,fDict[k]) for i,k in enumerate(fDict)}

    # 3. Output: FILE 
    keyFN = self.keyFN('key.pdf')
    ts = pdfFill(self.inFN(), keyFN)
    oDict = ts.fillPDF(to = testDict, )

    print(f"{self.__class__.__name__} SUCC: testDict Generated {len(testDict)} Fields; \n-- FILE: {Path(keyFN)}")

genTestKeyDict(fm)

Json FN /Users/frankrojas/GDrive/Family/Assets-Hobby/RealEstateInvestments/LLC-WB-Group/pages/AccountingData/2025/YE_Tax_Records/Forms_IRS/Schedule_K_1-FieldNames.json
Loading Form Field Name (json): Schedule_K_1-FieldNames.json
DIR: /Users/frankrojas/GDrive/Family/Assets-Hobby/RealEstateInvestments/LLC-WB-Group/pages/AccountingData/2025/YE_Tax_Records/Forms_IRS/Schedule_K_1-FieldNames.json
111 111
Loading Form Field Name (json): Schedule_K_1-FieldNames.json
DIR: /Users/frankrojas/GDrive/Family/Assets-Hobby/RealEstateInvestments/LLC-WB-Group/pages/AccountingData/2025/YE_Tax_Records/Forms_IRS/Schedule_K_1-FieldNames.json


TypeError: irsForms.keyFN() takes 1 positional argument but 2 were given

## Generate Test PDF :: testDict

In [61]:
fldNmDict = fm.fldNmLoad()
testDict =  {}
for (i,k1),(j,k2) in zip(enumerate(fDict), enumerate(fldNmDict)):
    d = fDict[k1]
    d['field_id'] = k2
    testDict[k1] = d
print(f"\nPDF Out w/ Keys: {Path(fm.outFN()).name} -- Total Fields: {len(fDict)}")    
list([(k,d) for k,d in testDict.items()])[0:3]


Loading Form Field Name (json): Schedule_K_1-FieldNames.json
DIR: /Users/frankrojas/GDrive/Family/Assets-Hobby/RealEstateInvestments/LLC-WB-Group/pages/AccountingData/2025/YE_Tax_Records/Forms_IRS/Schedule_K_1-FieldNames.json

PDF Out w/ Keys: Schedule_K_1-keys.pdf -- Total Fields: 111


[('topmostSubform[0].Page1[0].Pg1Header[0].ForCalendarYear[0].f1_1[0]',
  {'field_id': 'P1_Hdr_0', 'type': 'text', 'page': 1}),
 ('topmostSubform[0].Page1[0].Pg1Header[0].ForCalendarYear[0].f1_2[0]',
  {'field_id': 'P1_Hdr_1', 'type': 'text', 'page': 1}),
 ('topmostSubform[0].Page1[0].Pg1Header[0].ForCalendarYear[0].f1_3[0]',
  {'field_id': 'P1_Hdr_2', 'type': 'text', 'page': 1})]

In [6]:
# Generate default Test PDF
if False:
    pf = pdfFill(inFN, outFN)
    oDict = pf.fillPDF()
    print(f"\nPDF Out w/ Keys (defaults): {Path(fm.outFN()).name} -- Total Fields: {len(fDict)}")
    list([(k,d) for k,d in oDict.items()])[0:3]

# Generate Test Dict to map fields to Test Field name F#_P#
testDict = {k:pf._testKey(i,k,fDict[k]) for i,k in enumerate(fDict)}
print(f"\nPDF Out w/ Keys: {Path(fm.outFN()).name} -- Total Fields: {len(fDict)}")    
list([(k,d) for k,d in testDict.items()])[0:3]
    


PDF Out w/ Keys: Form_1065-keys.pdf -- Total Fields: 440


[('topmostSubform[0].Page1[0].HeaderAddress_ReadOrder[0].CalendarName_ReadOrder[0].f1_01[0]',
  {'field_id': 'F0', 'type': 'text', 'page': 1, 'value': ''}),
 ('topmostSubform[0].Page1[0].HeaderAddress_ReadOrder[0].CalendarName_ReadOrder[0].f1_02[0]',
  {'field_id': 'F1', 'type': 'text', 'page': 1, 'value': ''}),
 ('topmostSubform[0].Page1[0].HeaderAddress_ReadOrder[0].CalendarName_ReadOrder[0].f1_03[0]',
  {'field_id': 'F2', 'type': 'text', 'page': 1, 'value': ''})]

## Generate PDF Keys File :: oDict

Use keys.json to build GL dict


In [62]:
outFN = fm.outFN()
print("PDF Key file", outFN)
pf = pdfFill(fm.inFN(), outFN)
oDict = pf.fillPDF(to = testDict)
list([(k,d) for k,d in oDict.items()])[0:3]

PDF Key file /Users/frankrojas/GDrive/Family/Assets-Hobby/RealEstateInvestments/LLC-WB-Group/pages/AccountingData/2025/YE_Tax_Records/Forms_IRS/Schedule_K_1-keys.pdf


[('topmostSubform[0].Page1[0].Pg1Header[0].ForCalendarYear[0].f1_1[0]',
  'P1_Hdr_0'),
 ('topmostSubform[0].Page1[0].Pg1Header[0].ForCalendarYear[0].f1_2[0]',
  'P1_Hdr_1'),
 ('topmostSubform[0].Page1[0].Pg1Header[0].ForCalendarYear[0].f1_3[0]',
  'P1_Hdr_2')]

## Generate Keys JSON file to fill in

In [8]:
with open(fm.keyFN(), 'w') as fio:
    json.dump(testDict, fio, indent = 4)

In [9]:
glDict = fm.fm2glDict()
list([(k,d) for k,d in glDict.items()])[0:3]

AttributeError: 'irsF1065' object has no attribute 'fmDict'

# Generate glMapDict into json file

In [ ]:
print(len(glDict), len(testDict))
d = {}
for (i,k1),(j,k2)  in  zip(enumerate(glDict), enumerate(testDict)):
    g = glDict[k1]
    fldDict = testDict[k2]
    f = fldDict['field_id']
    if f != g : 
        print("ERROR: glDict and fmDict have a mismatch")    
        print("--ERR>", i,j, glDict[k1], testDict[k2])
    v = '' if fldDict['type'] == 'text' else 'chk'
    d[k1] = v
list([(k,d) for k,d in d.items()])[0:3]
fn = fm.FN('glKeys.json')
print(fn)
with open(fn, 'w') as fio:
    json.dump(d, fio, indent=4)